In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew

2025-04-10 00:37:02,847 - 21244 - __init__.py-__init__:362 - ERROR: Exception while exporting Span batch.
Traceback (most recent call last):
  File "c:\Users\91888\anaconda3\envs\crewai-env\lib\site-packages\urllib3\connectionpool.py", line 787, in urlopen
    response = self._make_request(
  File "c:\Users\91888\anaconda3\envs\crewai-env\lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "c:\Users\91888\anaconda3\envs\crewai-env\lib\site-packages\urllib3\connection.py", line 516, in getresponse
    httplib_response = super().getresponse()
  File "c:\Users\91888\anaconda3\envs\crewai-env\lib\http\client.py", line 1375, in getresponse
    response.begin()
  File "c:\Users\91888\anaconda3\envs\crewai-env\lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
  File "c:\Users\91888\anaconda3\envs\crewai-env\lib\http\client.py", line 287, in _read_status
    raise RemoteDisconnected("Remote en

In [19]:
import os
from utils.api_utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [6]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [9]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [10]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    agent=trading_strategy_agent,
)


In [11]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    agent=execution_agent,
)


In [12]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    agent=risk_management_agent,
)

Creating the Crew¶
The Process class helps to delegate the workflow to the Agents (kind of like a Manager at work)
In the example below, it will run this hierarchically.
manager_llm lets you choose the "manager" LLM you want to use.

In [13]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

In [14]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'AAPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

In [20]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6a9ae337-8a5a-49f4-ac40-28ad376b2ffe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 960b5af8-202d-4995-91ef-be222c32426b
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 960b5af8-202d-4995-91ef-be222c32426b
       Status: Executing Task...
    └── 🤖 Agent: Crew Manager
            Status: In Progress

# Agent: Crew Manager
## Task: Continuously monitor and analyze market data for the selected stock (AAPL). Use statistical modeling and machine learning to identify trends and predict market movements.


🚀 Crew: crew
└── 📋 Task: 960b5af8-202d-4995-91ef-be222c32426b
       Status: Executing Task...
    └── 🤖 Agent: Crew Manager
            Status: In Progress
        └── 🧠 Thinking...

...



# Agent: Crew Manager
## Final Answer: 
**Comprehensive Risk Analysis Report for AAPL Trading Strategies and Execution Plans**

**1. Introduction:**
This risk analysis report aims to evaluate the risks associated with the proposed trading strategies and execution plans for AAPL stock. By leveraging historical stock performance data, user-defined risk tolerance, and trading preferences, we can identify potential risks and recommend mitigation strategies to optimize trading outcomes.

**2. Historical Stock Performance Analysis:**
- AAPL's stock price has shown fluctuations over the years, with the current closing price at $181.46 as of April 7, 2025.
- The all-time high closing price was $258.74 on December 26, 2024, with a 52-week high and low of $260.10 and $164.08, respectively.
- The average stock price for the last 52 weeks stands at $220.53.

**3. Risk Tolerance and Trading Preferences:**
- Day trading involves significant risk due to leveraging capital daily. Individuals with li

🚀 Crew: crew
├── 📋 Task: 960b5af8-202d-4995-91ef-be222c32426b
│      Assigned to: Crew Manager
│      Status: ✅ Completed
│   ├── 🤖 Agent: Crew Manager
│   │       Status: In Progress
│   │   └── 🔧 Using Delegate work to coworker (3)
│   ├── 🤖 Agent: Crew Manager
│   │       Status: ✅ Completed
│   │   └── 🔧 Using Ask question to coworker (1)
│   └── 🤖 Agent: Crew Manager
│           Status: ✅ Completed
├── 📋 Task: d1d21cbb-0703-4579-901f-3efcf9865f04
│      Assigned to: Crew Manager
│      Status: ✅ Completed
│   └── 🤖 Agent: Crew Manager
│           Status: ✅ Completed
│       ├── 🔧 Used Delegate work to coworker (4)
│       ├── 🔧 Used Search the internet with Serper (9)
│       └── 🔧 Used Read website content (10)
├── 📋 Task: 58430ed0-19d5-496f-8c51-c252dbd3dea6
│      Assigned to: Crew Manager
│      Status: ✅ Completed
│   ├── 🤖 Agent: Crew Manager
│   │       Status: In Progress
│   │   ├── 🔧 Used Read website content (11)
│   │   ├── 🔧 Used Search the internet with Serper (10)
│   │   └── 🔧 Using Delegate work to coworker (5)
│   └── 🤖 Agent: Crew Manager
│           Status: ✅ Completed
└── 📋 Task: 666ead97-22e5-4310-bfc1-32e7103abaa6
       Status: Executing Task...
    ├── 🤖 Agent: Crew Manager
    │       Status: In Progress
    │   └── 🔧 Using Delegate work to coworker (6)
    ├── 🤖 Agent: Crew Manager
    │       Status: ✅ Completed
    │   ├── 🔧 Used Search the internet with Serper (11)
    │   └── 🔧 Using Delegate work to coworker (7)
    └── 🤖 Agent: Crew Manager
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 960b5af8-202d-4995-91ef-be222c32426b
│      Assigned to: Crew Manager
│      Status: ✅ Completed
│   ├── 🤖 Agent: Crew Manager
│   │       Status: In Progress
│   │   └── 🔧 Using Delegate work to coworker (3)
│   ├── 🤖 Agent: Crew Manager
│   │       Status: ✅ Completed
│   │   └── 🔧 Using Ask question to coworker (1)
│   └── 🤖 Agent: Crew Manager
│           Status: ✅ Completed
├── 📋 Task: d1d21cbb-0703-4579-901f-3efcf9865f04
│      Assigned to: Crew Manager
│      Status: ✅ Completed
│   └── 🤖 Agent: Crew Manager
│           Status: ✅ Completed
│       ├── 🔧 Used Delegate work to coworker (4)
│       ├── 🔧 Used Search the internet with Serper (9)
│       └── 🔧 Used Read website content (10)
├── 📋 Task: 58430ed0-19d5-496f-8c51-c252dbd3dea6
│      Assigned to: Crew Manager
│      Status: ✅ Completed
│   ├── 🤖 Agent: Crew Manager
│   │       Status: In Progress
│   │   ├── 🔧 Used Read website content (11)
│   │   ├── 🔧 Used Search the internet with Serper (10)
│   │   └── 🔧 Using Delegate work to coworker (5)
│   └── 🤖 Agent: Crew Manager
│           Status: ✅ Completed
└── 📋 Task: 666ead97-22e5-4310-bfc1-32e7103abaa6
       Assigned to: Crew Manager
       Status: ✅ Completed
    ├── 🤖 Agent: Crew Manager
    │       Status: In Progress
    │   └── 🔧 Using Delegate work to coworker (6)
    ├── 🤖 Agent: Crew Manager
    │       Status: ✅ Completed
    │   ├── 🔧 Used Search the internet with Serper (11)
    │   └── 🔧 Using Delegate work to coworker (7)
    └── 🤖 Agent: Crew Manager
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 666ead97-22e5-4310-bfc1-32e7103abaa6                                                                     │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6a9ae337-8a5a-49f4-ac40-28ad376b2ffe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
from IPython.display import Markdown
Markdown(result.raw)

**Comprehensive Risk Analysis Report for AAPL Trading Strategies and Execution Plans**

**1. Introduction:**
This risk analysis report aims to evaluate the risks associated with the proposed trading strategies and execution plans for AAPL stock. By leveraging historical stock performance data, user-defined risk tolerance, and trading preferences, we can identify potential risks and recommend mitigation strategies to optimize trading outcomes.

**2. Historical Stock Performance Analysis:**
- AAPL's stock price has shown fluctuations over the years, with the current closing price at $181.46 as of April 7, 2025.
- The all-time high closing price was $258.74 on December 26, 2024, with a 52-week high and low of $260.10 and $164.08, respectively.
- The average stock price for the last 52 weeks stands at $220.53.

**3. Risk Tolerance and Trading Preferences:**
- Day trading involves significant risk due to leveraging capital daily. Individuals with limited resources and low risk tolerance are not recommended for day trading.
- Risk management techniques like the 1% rule advise not risking more than 1% of the total account value on a single trade.
- Clear risk management strategies, position sizing, stop-loss orders, and leverage are vital for effective day trading risk management.

**4. Market Sentiment and News Analysis:**
- Recent news indicates positive sentiment towards AAPL, with the launch of a new product line driving sales growth.
- Market sentiment remains bullish, with analysts forecasting a potential increase in the stock price.
- Monitoring further developments and news is crucial to understanding the impact on AAPL's stock performance.

**5. Technical Analysis and Mitigation Strategies:**
- Utilize historical stock performance data and technical indicators like moving averages, RSI, and volume trends to assess AAPL's market conditions.
- Develop risk mitigation strategies based on historical price patterns, technical signals, and market dynamics.
- Implement stop-loss orders, position sizing techniques, and risk-reward ratios to manage trading risks effectively.

**6. Conclusion:**
In conclusion, a comprehensive approach that combines historical data analysis, risk management strategies, and market sentiment monitoring is essential for successful trading of AAPL stock. By aligning trading strategies with user-defined risk tolerance and preferences, traders can mitigate risks and optimize trading outcomes in the dynamic stock market environment.

This risk analysis report provides a detailed assessment of potential risks associated with the proposed trading strategies for AAPL and suggests mitigation strategies to enhance trading performance and minimize losses.